# ***Feature Engineering + EDA***

**Miêu tả:** Code này gộp pipeline FE + EDA cho cả 3 split (train/test/val) vào 1 notebook duy nhất. Nó đọc lần lượt 3 file CSV đầu vào, xử lý giá trị thiếu, rồi tạo 2 nhóm đặc trưng chính cho mỗi split:

* Review-level features: phân tích từng nội dung review để lấy dấu hiệu như từ khóa quảng cáo, link ngoài, thông tin liên hệ, dấu câu bất thường, chữ in hoa, emoji và độ lặp từ.
* User-level features: nhóm theo customer_id để tính hành vi người dùng như tần suất review, khoảng cách thời gian giữa các review, đăng review dồn dập, tuổi tài khoản, phương sai rating, tỷ lệ rating cực đoan và mức độ tập trung vào một seller.

* Sau khi tính features, code chạy EDA (thống kê mô tả, skewness, Mann-Whitney U test, Chi-square) và xuất 4 biểu đồ (KDE user-level, KDE review-level, binary flags, correlation heatmap) riêng cho từng split. Cuối cùng, user_features và review_features được lưu ra file CSV tương ứng để phục vụ bước huấn luyện mô hình.

# **Import + Config**

In [ ]:
import argparse, re, warnings, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

OUT = Path("output")
OUT.mkdir(exist_ok=True)
LABEL_COL = "label"

SPLITS = [
    ("train_clean.csv", "user_features.csv",     "review_features.csv"),
    ("test_clean.csv",  "user_features_test.csv", "review_features_test.csv"),
    ("val_clean.csv",   "user_features_val.csv",  "review_features_val.csv"),
]

sns.set_theme(style="whitegrid", font_scale=1.0)
PAL  = {0: "#4C9BE8", 1: "#E8694C"}
LMAP = {0: "Non-Suspicious", 1: "Suspicious"}

PROMO_KW = [
    "mua ngay","giảm giá","khuyến mãi","ưu đãi","sale","inbox",
    "liên hệ","order","freeship","voucher","discount","hot deal",
    "siêu rẻ","đặt hàng","combo","flash sale","giá tốt","giá sốc",
    "nhắn tin","tư vấn miễn phí",
]
PROMO_PAT      = "|".join(re.escape(k) for k in PROMO_KW)
URL_PAT        = r"https?://\S+|www\.\S+"
CONT_PAT       = r"(\b0[3-9]\d{8}\b)|(\+84\d{9,10})|(zalo|telegram|facebook|fb\.com|messenger)"
EMOJI_PAT      = re.compile(r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F900-\U0001F9FF]")
AVATAR_DEFAULT_PAT = r"//tiki\.vn/assets/img/avatar\.png"

def timer(msg, t0):
    print(f"  ✓ {msg:<50} {time.time()-t0:>6.1f}s")
    return time.time()

def save_fig(fig, name):
    fig.savefig(OUT / name, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {name}")

def _uwr(text):
    words = text.split()
    if len(words) == 0:
        return 0.0
    return 1.0 - len(set(words)) / len(words)

# **Main Loop**

In [ ]:
for CSV_PATH, OUT_USER, OUT_REVIEW in SPLITS:
    print("\n" + "="*65)
    print(f"  Processing: {CSV_PATH}")
    print("="*65)

    # ── 1. LOAD ──────────────────────────────────────────────────────
    t0 = time.time()
    raw = pd.read_csv(CSV_PATH, low_memory=False)

    print(f"  Shape: {raw.shape[0]:,} rows × {raw.shape[1]} cols")
    print(f"  Unique customer_id: {raw['customer_id'].nunique():,}")

    for col in ["customer_joined_time", "explain"]:
        if col in raw.columns:
            raw[col] = raw[col].replace(-999, np.nan)

    if "customer_joined_time" in raw.columns:
        raw["customer_joined_time"] = (
            raw["customer_joined_time"].astype(str)
            .str.extract(r'(\d+\.?\d*)', expand=False)
        )
        raw["customer_joined_time"] = pd.to_numeric(raw["customer_joined_time"], errors="coerce")

    if "rating"         in raw.columns: raw["rating"]         = pd.to_numeric(raw["rating"],         errors="coerce")
    if "content_length" in raw.columns: raw["content_length"] = pd.to_numeric(raw["content_length"], errors="coerce")
    if "content"        in raw.columns: raw["content"]        = raw["content"].replace("Missing", np.nan)

    raw["review_created_date"] = pd.to_datetime(raw["review_created_date"], errors="coerce")
    raw["review_created_date"] = raw["review_created_date"].clip(
        lower=pd.Timestamp("2000-01-01"),
        upper=pd.Timestamp("2030-12-31")
    )
    raw = raw.sort_values(["customer_id", "review_created_date"]).reset_index(drop=True)

    HAS_LABEL = LABEL_COL in raw.columns
    if HAS_LABEL:
        print(f"  Label dist: {raw[LABEL_COL].value_counts().to_dict()}")
    t0 = timer("Load & clean", t0)

    # ── 2. USER-LEVEL FEATURES ───────────────────────────────────────
    t0 = time.time()
    CID = "customer_id"

    rc  = raw.groupby(CID).size().rename("review_count_total").reset_index()
    dr  = raw.groupby(CID)["review_created_date"].agg(date_min="min", date_max="max").reset_index()
    dr["active_days"] = ((dr["date_max"] - dr["date_min"]).dt.days + 1).clip(lower=1)

    udf = dr.merge(rc, on=CID)
    udf["reviews_per_day"] = udf["review_count_total"] / udf["active_days"]

    rs         = raw[[CID, "review_created_date"]].copy()
    rs["prev"] = rs.groupby(CID)["review_created_date"].shift(1)
    rs["gap"]  = (rs["review_created_date"] - rs["prev"]).dt.total_seconds()
    tg         = rs.groupby(CID)["gap"].std().fillna(0).rename("time_gap_std").reset_index()
    udf        = udf.merge(tg, on=CID)

    raw["_date"] = raw["review_created_date"].dt.date
    md = (
        raw.groupby([CID, "_date"]).size()
        .groupby(level=0).max()
        .rename("max_reviews_in_one_day").reset_index()
    )
    raw.drop(columns=["_date"], inplace=True)
    udf = udf.merge(md, on=CID)

    aa = raw.groupby(CID)["customer_joined_time"].first().rename("account_age_days").reset_index()
    aa["account_age_days"] = aa["account_age_days"] * 30.44
    udf = udf.merge(aa, on=CID)

    rv = raw.groupby(CID)["rating"].var().rename("rating_variance").reset_index()
    udf = udf.merge(rv, on=CID)
    udf["rating_variance"] = udf["rating_variance"].fillna(0)

    raw["_extreme"] = ((raw["rating"] == 1) | (raw["rating"] == 5)).astype(int)
    pe = raw.groupby(CID)["_extreme"].mean().rename("percent_extreme_rating").reset_index()
    raw.drop(columns=["_extreme"], inplace=True)
    udf = udf.merge(pe, on=CID)

    sb     = raw.groupby([CID, "seller_id"]).size().reset_index(name="sc")
    sb_max = sb.groupby(CID)["sc"].max().rename("max_sc").reset_index()
    sb_df  = sb_max.merge(rc, on=CID)
    sb_df["single_brand_focus_ratio"] = sb_df["max_sc"] / sb_df["review_count_total"]
    udf    = udf.merge(sb_df[[CID, "single_brand_focus_ratio"]], on=CID)

    rlv = raw.groupby(CID)["content_length"].var().rename("review_length_variance").reset_index()
    udf = udf.merge(rlv, on=CID)

    if "avatar" in raw.columns:
        av = raw.groupby(CID)["avatar"].first().reset_index()
        av["avatar_default"] = (
            av["avatar"].str.contains(AVATAR_DEFAULT_PAT, na=True, regex=True)
            .map({True: 1, False: 0}).fillna(0).astype(int)
        )
        udf = udf.merge(av[[CID, "avatar_default"]], on=CID)
    else:
        udf["avatar_default"] = np.nan
        print("  [WARN] Cột 'avatar' không tìm thấy → avatar_default = NaN")

    if HAS_LABEL:
        lbl = raw.groupby(CID)[LABEL_COL].agg(lambda x: x.mode()[0]).rename("label").reset_index()
        udf = udf.merge(lbl, on=CID)
    else:
        udf["label"] = np.nan

    t0 = timer(f"User-level features → {len(udf):,} users", t0)

    # ── 3. REVIEW-LEVEL FEATURES ─────────────────────────────────────
    t0    = time.time()
    valid = raw["content"].notna() & (raw["content"].str.strip() != "")
    rdf   = raw[valid].copy()

    c       = rdf["content"].fillna("")
    n_words = c.str.split().str.len().clip(lower=1)
    n_chars = c.str.len().clip(lower=1)
    n_alpha = c.str.count(r"[A-Za-z]").clip(lower=1)

    rdf["promo_keyword_ratio"]   = c.str.lower().str.count(PROMO_PAT) / n_words
    rdf["external_link_flag"]    = c.str.contains(URL_PAT,  case=False, na=False).astype(int)
    rdf["contact_info_flag"]     = c.str.contains(CONT_PAT, case=False, na=False).astype(int)
    rdf["excessive_punctuation"] = c.str.count(r"[!?]") / n_chars
    rdf["capitalization_ratio"]  = c.str.count(r"[A-Z]") / n_alpha
    rdf["emoji_density"]         = c.apply(lambda x: len(EMOJI_PAT.findall(x))) / n_chars
    rdf["word_repetition_ratio"]     = c.apply(_uwr)

    if HAS_LABEL:
        rdf["label"] = rdf[LABEL_COL]

    t0 = timer(f"Review-level features → {len(rdf):,} reviews with content", t0)

    # ── 4. DESCRIPTIVE STATS + PLOTS (CHỈ cho tập train) ─────────────
    split_name = CSV_PATH.replace("_clean.csv", "")

    U_CONT = [
        "reviews_per_day", "time_gap_std", "max_reviews_in_one_day",
        "account_age_days", "rating_variance", "percent_extreme_rating",
        "single_brand_focus_ratio", "review_length_variance",
    ]
    U_BIN  = ["avatar_default"]
    R_CONT = [
        "promo_keyword_ratio", "excessive_punctuation", "capitalization_ratio",
        "emoji_density", "word_repetition_ratio",
    ]
    R_BIN  = ["external_link_flag", "contact_info_flag"]

    if split_name == "train":
        # ── 4a. Descriptive stats ────────────────────────────────────
        print("\n" + "="*72)
        print("  USER-LEVEL – describe()")
        print("="*72)
        print(udf[U_CONT].describe().T[["count","mean","std","min","50%","max"]].to_string())

        if udf["avatar_default"].notna().any():
            print(f"\n  avatar_default dist: {udf['avatar_default'].value_counts().to_dict()}")

        print("\n" + "="*72)
        print("  REVIEW-LEVEL – describe()")
        print("="*72)
        print(rdf[R_CONT + R_BIN].describe().T[["count","mean","std","min","50%","max"]].to_string())

        # ── 4b. Skewness ─────────────────────────────────────────────
        print("\n" + "-"*72)
        print("  SKEWNESS")
        print("-"*72)
        for col in U_CONT + R_CONT:
            src  = udf if col in U_CONT else rdf
            sk   = src[col].skew()
            note = ">>> RIGHT-SKEWED" if sk > 1 else (">>> LEFT-SKEWED" if sk < -1 else "OK")
            print(f"  {col:<35} {sk:>+8.3f}  {note}")

        # ── 4c. Stat tests ───────────────────────────────────────────
        if HAS_LABEL:
            print("\n" + "="*72)
            print("  MANN-WHITNEY U TEST – user-level")
            print("="*72)
            for col in U_CONT:
                g0 = udf.loc[udf["label"] == 0, col].dropna()
                g1 = udf.loc[udf["label"] == 1, col].dropna()
                if len(g0) > 1 and len(g1) > 1:
                    u, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
                    sig  = "*** SIG" if p < 0.05 else "—"
                    print(f"  {col:<35} p={p:.5f}  {sig}")

            print("\n  CHI-SQUARE – binary flags")
            for col in U_BIN + R_BIN:
                src = udf if col in U_BIN else rdf
                if src[col].notna().any():
                    ct = pd.crosstab(src["label"], src[col])
                    if ct.shape == (2, 2):
                        chi2, p, _, _ = stats.chi2_contingency(ct)
                        sig = "*** SIG" if p < 0.05 else "—"
                        print(f"  {col:<35} chi2={chi2:.3f}  p={p:.5f}  {sig}")

        # ── 4d. Helper functions plots ───────────────────────────────
        def kde_ax(ax, df, col, title, use_log=False, clip_q=0.995):
            if HAS_LABEL and "label" in df.columns:
                for lv in [0, 1]:
                    sub  = df.loc[df["label"] == lv, col].dropna()
                    vals = np.log1p(sub) if use_log else sub.clip(upper=sub.quantile(clip_q))
                    if len(vals) > 1:
                        sns.kdeplot(vals, ax=ax, label=LMAP[lv],
                                    color=PAL[lv], fill=True, alpha=0.3, linewidth=2)
                ax.legend(fontsize=8)
            else:
                sub  = df[col].dropna()
                vals = np.log1p(sub) if use_log else sub.clip(upper=sub.quantile(clip_q))
                if len(vals) > 1:
                    sns.kdeplot(vals, ax=ax, color="#4C9BE8", fill=True, alpha=0.4, linewidth=2)
            ax.set_title(title, fontsize=10, fontweight="bold")
            ax.set_xlabel("log1p(value)" if use_log else "value", fontsize=8)
            ax.set_ylabel("Density", fontsize=8)
            sns.despine(ax=ax)

        def binary_bar_ax(ax, df, col, title, label_0, label_1):
            counts = df[col].dropna().value_counts().reindex([0, 1], fill_value=0)
            colors = ["#4C9BE8", "#E8694C"]
            total  = counts.sum()
            bars   = ax.bar([label_0, label_1], counts.values, color=colors,
                            edgecolor="white", width=0.5, zorder=3)
            for b, v in zip(bars, counts.values):
                pct = v / total * 100 if total > 0 else 0
                ax.text(b.get_x() + b.get_width()/2, v + total*0.012,
                        f"{v:,}\n({pct:.1f}%)", ha="center", va="bottom",
                        fontsize=10, fontweight="bold")
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.set_ylabel("Số lượng", fontsize=9)
            ax.set_ylim(0, counts.max() * 1.20)
            ax.grid(axis="y", alpha=0.4, zorder=0)
            sns.despine(ax=ax)

        panels_u = [
            ("reviews_per_day",          "reviews_per_day\n= total_reviews / active_days",    False),
            ("time_gap_std",             "time_gap_std [giây]\n= std khoảng cách review",     False),
            ("max_reviews_in_one_day",   "max_reviews_in_one_day\n= max review trong 1 ngày", False),
            ("account_age_days",         "account_age_days\n= joined_time × 30.44",           False),
            ("rating_variance",          "rating_variance\n= var(rating) per customer",       False),
            ("percent_extreme_rating",   "percent_extreme_rating\n= (1s+5s)/total",           False),
            ("single_brand_focus_ratio", "single_brand_focus_ratio\n= max(per seller)/total", False),
            ("review_length_variance",   "review_length_variance\n= var(content_length)",     False),
        ]
        panels_r = [
            ("promo_keyword_ratio",   "promo_keyword_ratio\n= promo_kw_count / n_words"),
            ("excessive_punctuation", "excessive_punctuation\n= #(!?) / content_length"),
            ("capitalization_ratio",  "capitalization_ratio\n= #uppercase / #alpha"),
            ("emoji_density",         "emoji_density\n= #emoji / content_length"),
            ("word_repetition_ratio",     "word_repetition_ratio\n= 1 - unique/total words"),
        ]

        # Fig 1 – KDE user-level
        fig, axes = plt.subplots(3, 3, figsize=(17, 13))
        for ax_i, (col, title, do_log) in zip(axes.flatten(), panels_u):
            kde_ax(ax_i, udf, col, title, use_log=do_log)
        axes.flatten()[-1].set_visible(False)
        fig.suptitle(f"KDE – User-level Features [{split_name}]", fontsize=14, fontweight="bold")
        plt.tight_layout()
        save_fig(fig, f"fig_01_user_level_kde_{split_name}.png")

        # Fig 2 – KDE review-level
        fig, axes = plt.subplots(2, 3, figsize=(17, 9))
        for ax_i, (col, title) in zip(axes.flatten(), panels_r):
            kde_ax(ax_i, rdf, col, title)
        axes.flatten()[-1].set_visible(False)
        fig.suptitle(f"KDE – Review-level Features [{split_name}]", fontsize=14, fontweight="bold")
        plt.tight_layout()
        save_fig(fig, f"fig_02_review_level_kde_{split_name}.png")

        # Fig 3 – Binary flags
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        binary_bar_ax(axes[0], udf, "avatar_default",
                      "avatar_default\n(user-level)",
                      "Avatar mặc định\n(= 0)", "Có avatar riêng\n(= 1)")
        binary_bar_ax(axes[1], rdf, "external_link_flag",
                      "external_link_flag\n(review-level, có content)",
                      "Không có link\n(= 0)", "Có link ngoài\n(= 1)")
        binary_bar_ax(axes[2], rdf, "contact_info_flag",
                      "contact_info_flag\n(review-level, có content)",
                      "Không có TTLL\n(= 0)", "Có thông tin\nliên lạc (= 1)")
        fig.suptitle(f"Binary Flags [{split_name}]", fontsize=13, fontweight="bold")
        plt.tight_layout()
        save_fig(fig, f"fig_03_binary_flags_{split_name}.png")

        # Fig 4 – Correlation heatmap
        hcols = U_CONT + (["label"] if HAS_LABEL else [])
        corr  = udf[hcols].corr()
        fig, ax = plt.subplots(figsize=(12, 9))
        sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
                    annot=True, fmt=".2f", cmap="RdYlGn",
                    center=0, vmin=-1, vmax=1, linewidths=0.4,
                    annot_kws={"size": 8}, ax=ax)
        ax.set_title(f"Correlation – User-level Features [{split_name}]",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        save_fig(fig, f"fig_04_correlation_{split_name}.png")

    else:
        # Val/Test: bỏ qua toàn bộ stats, skewness, plots
        print(f"  [{split_name}] Bỏ qua descriptive stats, skewness và plots.")

    # ── 5. EXPORT ─────────────────────────────────────────────────────
    udf.to_csv(OUT / OUT_USER,   index=False)
    rdf.to_csv(OUT / OUT_REVIEW, index=False)
    print(f"\n  Saved: {OUT_USER:<35} ({len(udf):,} users × {udf.shape[1]} cols)")
    print(f"  Saved: {OUT_REVIEW:<35} ({len(rdf):,} reviews × {rdf.shape[1]} cols)")
    print(f"  Hoàn tất: {CSV_PATH}\n")


  Processing: train_clean.csv
  Shape: 409,333 rows × 18 cols
  Unique customer_id: 257,044
  ✓ Load & clean                                          5.2s
  ✓ User-level features → 257,044 users                   1.5s
  ✓ Review-level features → 142,920 reviews with content    7.3s

  USER-LEVEL – describe()
                             count          mean           std       min      50%           max
reviews_per_day           257044.0  8.701191e-01  5.539351e-01  0.000914     1.00  2.000000e+01
time_gap_std              257044.0  1.525325e+06  6.348003e+06  0.000000     0.00  1.212021e+08
max_reviews_in_one_day    257044.0  1.162365e+00  5.298213e-01  1.000000     1.00  2.300000e+01
account_age_days          257044.0  2.598609e+03  2.249516e+03  0.000000  2556.96  7.510157e+05
rating_variance           257044.0  7.222370e-02  5.322854e-01  0.000000     0.00  8.000000e+00
percent_extreme_rating    257044.0  8.595947e-01  3.304091e-01  0.000000     1.00  1.000000e+00
single_brand_focu